In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data.load_data import load_ratings
from src.models.neural_network import NeuralNetworkRecommender

DATA_DIR = PROJECT_ROOT / "data" / "raw"

I0000 00:00:1793457378.621461   15750 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793457378.645881   15750 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793457384.410384   15750 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793457390.209380   15750 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [3]:
ratings = load_ratings(DATA_DIR)

ratings.head()

,user,item,rating
0,1889878,CC0101EN,3.0
1,1342067,CL0101EN,3.0
2,1990814,ML0120ENv3,3.0
3,380098,BD0211EN,3.0
4,779563,DS0101EN,3.0


In [4]:
user_ids = ratings["user"].unique()
item_ids = ratings["item"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

item_to_index = {
    item_id: index
    for index, item_id in enumerate(item_ids)
}

ratings_nn = ratings.copy()

ratings_nn["user_index"] = ratings_nn["user"].map(user_to_index)
ratings_nn["item_index"] = ratings_nn["item"].map(item_to_index)

ratings_nn.head()

,user,item,rating,user_index,item_index
0,1889878,CC0101EN,3.0,0,0
1,1342067,CL0101EN,3.0,1,1
2,1990814,ML0120ENv3,3.0,2,2
3,380098,BD0211EN,3.0,3,3
4,779563,DS0101EN,3.0,4,4


In [5]:
X_users = ratings_nn["user_index"].to_numpy()
X_items = ratings_nn["item_index"].to_numpy()
y = ratings_nn["rating"].to_numpy(dtype=np.float32)

print(f"Users: {len(user_to_index):,}")
print(f"Items: {len(item_to_index):,}")
print(f"Interactions: {len(y):,}")

Users: 33,901
Items: 126
Interactions: 233,306


In [6]:
from sklearn.model_selection import train_test_split

train_users, test_users, train_items, test_items, train_ratings, test_ratings = (
    train_test_split(
        X_users,
        X_items,
        y,
        test_size=0.20,
        random_state=42,
    )
)

print(f"Training interactions: {len(train_ratings):,}")
print(f"Testing interactions: {len(test_ratings):,}")

Training interactions: 186,644
Testing interactions: 46,662


In [7]:
nn_recommender = NeuralNetworkRecommender(
    n_users=len(user_to_index),
    n_items=len(item_to_index),
    embedding_dim=16,
)

nn_model = nn_recommender.build_model()

nn_model.summary()

E0000 00:00:1793457849.462139   15750 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_id             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_id             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 16)     │    542,416 │ user_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 16)     │      2,016 │ item_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 16)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 16)        │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 1)         │          0 │ dot[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 544,432 (2.08 MB)

 Trainable params: 544,432 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = nn_model.fit(
    [train_users, train_items],
    train_ratings,
    validation_data=(
        [test_users, test_items],
        test_ratings,
    ),
    epochs=10,
    batch_size=256,
    verbose=1,
)

Epoch 1/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 7.9868 - mae: 2.8072 - val_loss: 4.9247 - val_mae: 2.1700
Epoch 2/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 1.7612 - mae: 1.0974 - val_loss: 0.6715 - val_mae: 0.6118
Epoch 3/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.2509 - mae: 0.3277 - val_loss: 0.2685 - val_mae: 0.2851
Epoch 4/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0665 - mae: 0.1425 - val_loss: 0.1903 - val_mae: 0.1821
Epoch 5/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.0225 - mae: 0.0771 - val_loss: 0.1688 - val_mae: 0.1435
Epoch 6/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0108 - mae: 0.0523 - val_loss: 0.1619 - val_mae: 0.1295
Epoch 7/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0079 - mae: 0.0475 - val_loss: 0.1604 - val_mae: 0.1316
Epoch 8/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0091 - mae: 0.0595 - val_loss: 0.1629 - val_mae: 0.1470
Epoch 9/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - lo

In [9]:
test_loss, test_mae = nn_model.evaluate(
    [test_users, test_items],
    test_ratings,
    verbose=0,
)

nn_rmse = np.sqrt(test_loss)

print(f"Neural Network Test RMSE: {nn_rmse:.4f}")
print(f"Neural Network Test MAE: {test_mae:.4f}")

Neural Network Test RMSE: 0.4076
Neural Network Test MAE: 0.1617


In [10]:
nn_model.save(
    PROJECT_ROOT / "models" / "neural_network_recommender.keras"
)

In [12]:
# Extract and save the learned embedding matrices
user_embeddings = nn_recommender.user_embedding.get_weights()[0]
item_embeddings = nn_recommender.item_embedding.get_weights()[0]

np.save(
    PROJECT_ROOT / "models" / "user_embeddings.npy",
    user_embeddings
)

np.save(
    PROJECT_ROOT / "models" / "item_embeddings.npy",
    item_embeddings
)

print(f"User embedding shape: {user_embeddings.shape}")
print(f"Item embedding shape: {item_embeddings.shape}")

User embedding shape: (33901, 16)
Item embedding shape: (126, 16)
